# 🌪️ NCSSM WindTunnel lab · 2D Lattice Boltzmann Wind Tunnel
**North Carolina School of Science and Mathematics**  
*Real-Time Computational Aerodynamics · LBM D2Q9 · GPU Acceleration & WebGL Visualization*

---

### ⚡ Quickstart Instructions for Google Colab:
1. **Enable Free GPU Acceleration (Recommended)**:
   * Go to **Runtime** ➔ **Change runtime type** in the top menu.
   * Select **T4 GPU** (or any available GPU) under *Hardware accelerator* and click **Save**.
2. **Run Step 1**: Sets up the Python environment, CuPy GPU backend, and tunneling utilities.
3. **Run Step 2**: Launches the simulation backend and creates a secure Cloudflare tunnel.
4. Click **`🚀 Open WindTunnel lab ↗`** to open the interactive wind tunnel in a new tab!
*(If your campus or school network blocks Cloudflare, use **Step 2 (Alternative)** below)*.

In [ ]:
# @title Step 1: Clone Repository & Setup Environment { display-mode: "form" }
import os, sys, subprocess

print("🚀 [1/3] Setting up NCSSM WindTunnel lab workspace...")

# Locate or clone repository into workspace
if not os.path.exists("backend"):
    if not os.path.exists("WindTunnel"):
        !git clone https://github.com/fsantibanezc/WindTunnel.git
    if os.path.exists("WindTunnel"):
        %cd WindTunnel

print("📦 [2/3] Installing Python dependencies (FastAPI, Uvicorn, CuPy CUDA 12)..." )
!pip install -q fastapi "uvicorn[standard]" numpy "cupy-cuda12x>=13.0"

print("🌐 [3/3] Installing Cloudflare Tunnel client...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Optional fallback tunnel utility for networks that filter trycloudflare
!npm install -g localtunnel > /dev/null 2>&1 || true

print("\n✅ Environment ready! Now run Step 2 below.")

In [ ]:
# @title Step 2: Launch WindTunnel lab Server & Tunnel { display-mode: "form" }
import subprocess, time, re, os, sys, socket
from IPython.display import display, HTML

# 1. Clean up any existing instances
!pkill -f "backend/server.py" > /dev/null 2>&1 || true
!pkill -f "cloudflared" > /dev/null 2>&1 || true
time.sleep(1)

# Ensure we are in the project directory containing backend/
if not os.path.exists("backend") and os.path.exists("WindTunnel"):
    os.chdir("WindTunnel")

print("🌪️ Starting WindTunnel lab simulation server (logging to server.log)...")
server_log = open("server.log", "w")
server_proc = subprocess.Popen(
    [sys.executable, "backend/server.py"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# 2. Verify server successfully bound to port 8000 before launching tunnel
server_ok = False
for _ in range(30):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        if s.connect_ex(("127.0.0.1", 8000)) == 0:
            server_ok = True
            break
    time.sleep(0.5)

if not server_ok:
    print("❌ Server failed to bind to port 8000. Displaying server.log:")
    with open("server.log") as f:
        print(f.read())
    raise RuntimeError("Server initialization failed. See log output above.")

print("✅ Local server is live on port 8000.")
print("🔗 Connecting Cloudflare Tunnel via HTTP/2 (logging to tunnel.log)...")

# Use --protocol http2 to prevent UDP/QUIC drops and write directly to disk log
tunnel_log = open("tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--protocol", "http2"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# 3. Poll tunnel.log until the trycloudflare URL appears
tunnel_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print(f"\n🔗 Assigned URL: {tunnel_url}")
    print("⏳ Waiting 6 seconds for Cloudflare edge DNS propagation...")
    time.sleep(6)
    print("✨ Ready!")

    html_banner = f"""
    <div style="background: linear-gradient(135deg, #09121d, #142236); border: 1.5px solid #00e5ff; border-radius: 12px; padding: 22px; margin: 12px 0; color: #ffffff; box-shadow: 0 8px 32px rgba(0, 229, 255, 0.25); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
      <div style="display: flex; align-items: center; gap: 16px; margin-bottom: 12px;">
        <div style="background: #ffffff; border-radius: 8px; padding: 4px; display: flex; align-items: center; justify-content: center; width: 48px; height: 48px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);">
          <img src="{tunnel_url}/static/ncssm_logo.jpg" alt="NCSSM Logo" style="width: 40px; height: 40px; object-fit: contain;" />
        </div>
        <div>
          <h2 style="margin: 0; color: #00e5ff; font-size: 20px; letter-spacing: 0.5px;">NCSSM WindTunnel lab</h2>
          <p style="margin: 2px 0 0; color: #94a3b8; font-size: 13px;">2D Lattice Boltzmann Wind Tunnel · Live WebGL & GPU</p>
        </div>
      </div>
      <p style="color: #cbd5e1; font-size: 14px; margin: 14px 0 16px;">
        Your high-performance simulation instance is live and streaming at full GPU speed. Click below to launch the wind tunnel in a new tab:
      </p>
      <a href="{tunnel_url}" target="_blank" style="display: inline-block; background: linear-gradient(135deg, #00e5ff, #c084fc); color: #061018; font-weight: 700; font-size: 15px; padding: 12px 26px; border-radius: 8px; text-decoration: none; box-shadow: 0 4px 18px rgba(0, 229, 255, 0.4);">
        🚀 Open WindTunnel lab ↗
      </a>
      <div style="margin-top: 14px; font-size: 12px; color: #64748b;">
        Direct link: <a href="{tunnel_url}" target="_blank" style="color: #38bdf8;">{tunnel_url}</a>
      </div>
    </div>
    """
    display(HTML(html_banner))
else:
    print("❌ Could not obtain public tunnel URL. Inspecting tunnel.log:")
    if os.path.exists("tunnel.log"):
        with open("tunnel.log") as f:
            print(f.read())


In [ ]:
# @title Step 2 (Alternative): Launch via Localtunnel (use if Cloudflare is blocked by network) { display-mode: "form" }
import subprocess, time, re, os, urllib.request
from IPython.display import display, HTML

!pkill -f "localtunnel" > /dev/null 2>&1 || true

# Obtain external IP for Localtunnel password prompt
try:
    ip_pwd = urllib.request.urlopen("https://loca.lt/mytunnelpassword", timeout=5).read().decode("utf8").strip()
except Exception:
    ip_pwd = "See cell output"

lt_log = open("localtunnel.log", "w")
lt_proc = subprocess.Popen(
    ["npx", "-y", "localtunnel", "--port", "8000"],
    stdout=lt_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

lt_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists("localtunnel.log"):
        with open("localtunnel.log") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.loca\.lt", content)
            if match:
                lt_url = match.group(0)
                break

if lt_url:
    display(HTML(f"""
    <div style="background: #09121d; border: 1.5px solid #c084fc; border-radius: 12px; padding: 20px; margin: 12px 0; color: #ffffff;">
      <h3 style="color: #c084fc; margin-top: 0;">🌐 Localtunnel Active</h3>
      <p>Click to open: <a href="{lt_url}" target="_blank" style="color: #38bdf8; font-weight: bold; font-size: 16px;">{lt_url} ↗</a></p>
      <p style="font-size: 13px; color: #94a3b8; margin-top: 8px;">
        <b>Important:</b> Localtunnel displays an initial security prompt asking for your <i>Endpoint IP</i>. Enter this password: <code style="background: #1e293b; color: #facc15; padding: 3px 8px; border-radius: 4px; font-weight: bold;">{ip_pwd}</code>
      </p>
    </div>
    """))
else:
    print("❌ Failed to start localtunnel. Check localtunnel.log:")
    if os.path.exists("localtunnel.log"):
        with open("localtunnel.log") as f:
            print(f.read())


In [ ]:
# @title Step 3: Diagnostic Health & Live Logs { display-mode: "form" }
!echo "=== Active Processes ==="
!ps aux | grep -E "server.py|cloudflared|localtunnel" | grep -v grep || echo "No active background processes found."
!echo "\n=== Last 15 lines of server.log ==="
!tail -n 15 server.log 2>/dev/null || echo "No server.log yet."
!echo "\n=== Last 15 lines of tunnel.log ==="
!tail -n 15 tunnel.log 2>/dev/null || echo "No tunnel.log yet."
